In [1]:
import os, sys, subprocess, numpy as np, pandas as pd
from pathlib import Path
from typing import Tuple, Dict

RDKit_WHL_DIR = "/kaggle/input/rdkit-2025-3-3-cp311"
MORDRED_DIR   = "/kaggle/input/mordred-1-2-0-py3-none-any"
RDKIT_WHL     = f"{RDKit_WHL_DIR}/rdkit-2025.3.3-cp311-cp311-manylinux_2_28_x86_64.whl"
MORDRED_WHL   = f"{MORDRED_DIR}/mordred-1.2.0-py3-none-any.whl"
NX_WHL        = f"{MORDRED_DIR}/networkx-2.8.8-py3-none-any.whl"

def _pip(path, *extra):
    if os.path.exists(path):
        print(f"[Install] {os.path.basename(path)} {' '.join(extra)}")
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *extra, path])

_pip(RDKIT_WHL)
_pip(NX_WHL)
_pip(MORDRED_WHL, "--no-deps")

from rdkit import Chem
from mordred import Calculator, descriptors
from catboost import CatBoostRegressor
from sklearn.model_selection import KFold
from sklearn.metrics import mean_absolute_error

def _find_data_dir() -> str:
    for d in Path("/kaggle/input").glob("*"):
        if d.is_dir():
            name = d.name.lower()
            if any(k in name for k in ["polymer","neurips","open","opp"]) and (d/"train.csv").exists() and (d/"test.csv").exists():
                return str(d)
    return "/kaggle/input/neurips-open-polymer-prediction-2025"

DATA_DIR = _find_data_dir()
DESC_DIR = "/kaggle/input/modred-dataset"
print("[DATA_DIR]", DATA_DIR)

train = pd.read_csv(f"{DATA_DIR}/train.csv")
test  = pd.read_csv(f"{DATA_DIR}/test.csv")

def _find_smiles(df: pd.DataFrame) -> str:
    for c in df.columns:
        if str(c).lower().strip() in ("smiles","smile","smiles_str","polymer_smiles"):
            return c
    raise KeyError("SMILES column not found in train.csv")
SMI_COL = _find_smiles(train)

TARGETS = ["Tg","FFV","Tc","Density","Rg"]

tg      = pd.read_csv(f"{DESC_DIR}/desc_tg.csv",      low_memory=False)
tc      = pd.read_csv(f"{DESC_DIR}/desc_tc.csv",      low_memory=False)
rg      = pd.read_csv(f"{DESC_DIR}/desc_rg.csv",      low_memory=False)
ffv     = pd.read_csv(f"{DESC_DIR}/desc_ffv.csv",     low_memory=False)
density = pd.read_csv(f"{DESC_DIR}/desc_de.csv",      low_memory=False)

def _prep_train_df(df: pd.DataFrame, target: str) -> Tuple[pd.DataFrame, pd.Series]:
    smiles = df["SMILES"].astype(str) if "SMILES" in df.columns else pd.Series([None]*len(df))
    const = [c for c in df.columns if df[c].nunique(dropna=True) == 1]
    if const:
        df = df.drop(columns=const)
    y = pd.to_numeric(df[target], errors="coerce")
    X = df.drop(columns=[target], errors="ignore").select_dtypes(exclude=["object","category"])
    X[target] = y
    X = X.replace([np.inf, -np.inf], np.nan)
    return X, smiles

tg, tg_smiles             = _prep_train_df(tg, "Tg")
ffv, ffv_smiles           = _prep_train_df(ffv, "FFV")
tc, tc_smiles             = _prep_train_df(tc, "Tc")
density, density_smiles   = _prep_train_df(density, "Density")
rg, rg_smiles             = _prep_train_df(rg, "Rg")

print("tg shape:", tg.shape)
print("ffv shape:", ffv.shape)
print("tc shape:", tc.shape)
print("density shape:", density.shape)
print("rg shape:", rg.shape)

print("[Mordred] Computing test descriptors (2D; ignore_3D=True)...")
mols_test = [Chem.MolFromSmiles(s) for s in test[SMI_COL].astype(str)]
desc_test_raw = Calculator(descriptors, ignore_3D=True).pandas(mols_test)
desc_test_raw = desc_test_raw.apply(pd.to_numeric, errors="coerce").replace([np.inf, -np.inf], np.nan)
print("desc_test shape:", desc_test_raw.shape)

def _align(train_df, test_desc, target):
    feat_cols = sorted((set(train_df.columns) - {target}) & set(test_desc.columns))
    X  = train_df[feat_cols].copy()
    y  = train_df[target].copy()
    Xt = test_desc[feat_cols].copy()
    return X, y, Xt, feat_cols

RANDOM_SEEDS = [42, 2025, 777]
N_SPLITS = 5

def mordred_oof_and_test(train_df: pd.DataFrame,
                         smiles_series: pd.Series,
                         test_desc: pd.DataFrame,
                         target: str):
    keep = train_df[target].notna()
    tr_df = train_df.loc[keep].reset_index(drop=True)
    tr_smiles = smiles_series.loc[keep].reset_index(drop=True)  # alignment handle
    X, y, Xt, feat_cols = _align(tr_df, test_desc, target)

    kf = KFold(n_splits=N_SPLITS, shuffle=True, random_state=123)
    oof = np.zeros(len(X), dtype=float)

    for fold, (tr, va) in enumerate(kf.split(X), 1):
        model = CatBoostRegressor(
            loss_function="MAE",
            depth=8,
            learning_rate=0.05,
            iterations=2000,
            od_type="Iter",
            od_wait=200,
            random_seed=42,
            verbose=False,
        )
        model.fit(X.iloc[tr], y.iloc[tr],
                  eval_set=(X.iloc[va], y.iloc[va]),
                  use_best_model=True,
                  verbose=False)
        oof[va] = model.predict(X.iloc[va])

    oof_mae = mean_absolute_error(y, oof)
    print(f"  [{target}] Mordred OOF MAE = {oof_mae:.6f}  (features: {len(feat_cols)})")

    preds = []
    for seed in RANDOM_SEEDS:
        m = CatBoostRegressor(
            loss_function="MAE",
            depth=8,
            learning_rate=0.05,
            iterations=2000,
            od_type="Iter",
            od_wait=200,
            random_seed=seed,
            verbose=False,
        )
        m.fit(X, y, eval_set=(X, y), use_best_model=True, verbose=False)
        preds.append(m.predict(Xt))
    pred_test = np.mean(np.vstack(preds), axis=0)
    return oof, pred_test, tr_smiles

desc_test = desc_test_raw.copy()
OOF_MORDRED, TEST_MORDRED, SMILES_NONNULL = {}, {}, {}
for (df, smi, t) in [(tg, tg_smiles, "Tg"),
                     (ffv, ffv_smiles, "FFV"),
                     (tc, tc_smiles, "Tc"),
                     (density, density_smiles, "Density"),
                     (rg, rg_smiles, "Rg")]:
    oof, pred, sm = mordred_oof_and_test(df, smi, desc_test, t)
    OOF_MORDRED[t]  = oof
    TEST_MORDRED[t] = pred
    SMILES_NONNULL[t] = sm

def _discover_inputs():
    chem_root = None
    fast_root = None
    chem_sub = None
    fast_sub = None
    base = "/kaggle/input"
    for slug in os.listdir(base):
        slug_path = os.path.join(base, slug)
        if not os.path.isdir(slug_path):
            continue
        for r, d, f in os.walk(slug_path):
            fl = [x.lower() for x in f]
            if "chemberta_submission.csv" in fl:
                chem_sub = os.path.join(r, "chemberta_submission.csv")
            if "fastsmiles_submission.csv" in fl:
                fast_sub = os.path.join(r, "fastsmiles_submission.csv")
            if any(x.startswith("chemberta_a_") and x.endswith("_oof_preds.csv") for x in fl):
                chem_root = slug_path
            if any(x in ("ctb_a_oof_preds.csv","lgb_a_oof_preds.csv","xgb_a_oof_preds.csv") for x in fl):
                fast_root = slug_path
    return chem_root, fast_root, chem_sub, fast_sub

CHEMBERTA_ROOT, FASTSMILES_ROOT, CHEMBERTA_SUB, FASTSMILES_SUB = _discover_inputs()
print("[AUTO] CHEMBERTA_ROOT:", CHEMBERTA_ROOT)
print("[AUTO] FASTSMILES_ROOT:", FASTSMILES_ROOT)
print("[AUTO] CHEMBERTA_SUB:", CHEMBERTA_SUB)
print("[AUTO] FASTSMILES_SUB:", FASTSMILES_SUB)

def _align_oof_by_id(oof_csv_path: str, target_name: str) -> np.ndarray:
    """Return a full-length OOF vector aligned to train['id'] if possible."""
    df_oof = pd.read_csv(oof_csv_path)
    if "id" in df_oof.columns and "oof_preds" in df_oof.columns:
        series = pd.Series(df_oof["oof_preds"].values, index=df_oof["id"].values)
        return series.reindex(train["id"].values).values
    if "id" in df_oof.columns and target_name in df_oof.columns:
        series = pd.Series(df_oof[target_name].values, index=df_oof["id"].values)
        return series.reindex(train["id"].values).values
    for c in df_oof.columns:
        if c.lower() != "id" and pd.api.types.is_numeric_dtype(df_oof[c]):
            return df_oof[c].values
    return np.full(len(train), np.nan, dtype=float)

def _align_submission_to_test(df_sub: pd.DataFrame, target_name: str) -> np.ndarray:
    """Return test preds aligned to test['id'] order."""
    if "id" in df_sub.columns and target_name in df_sub.columns:
        s = pd.Series(df_sub[target_name].values, index=df_sub["id"].values)
        return s.reindex(test["id"].values).values
    return df_sub[target_name].values

def _maybe_load_oof_chemberta_auto() -> Dict[str, np.ndarray]:
    out = {}
    if not CHEMBERTA_ROOT or not os.path.exists(CHEMBERTA_ROOT):
        return out
    for t in TARGETS:
        base = None
        for d in os.listdir(CHEMBERTA_ROOT):
            p = os.path.join(CHEMBERTA_ROOT, d)
            if os.path.isdir(p) and t.lower() in d.lower():
                base = p; break
        if base is None:
            continue
        # pick a csv with "oof" in name
        cand = None
        for f in os.listdir(base):
            if f.lower().endswith(".csv") and "oof" in f.lower():
                cand = os.path.join(base, f); break
        if cand:
            out[t] = _align_oof_by_id(cand, t)
    return out

def _maybe_load_oof_fastsmiles_auto(weights=(0.4,0.3,0.3)) -> Dict[str, np.ndarray]:
    out = {}
    if not FASTSMILES_ROOT or not os.path.exists(FASTSMILES_ROOT):
        return out
    for t in TARGETS:
        tdir = os.path.join(FASTSMILES_ROOT, t)
        if not os.path.isdir(tdir):
            continue
        parts = {}
        for sub in os.listdir(tdir):
            base = os.path.join(tdir, sub)
            if not os.path.isdir(base):
                continue
            csv = None
            for f in os.listdir(base):
                if f.lower().endswith(".csv") and "oof" in f.lower():
                    csv = os.path.join(base, f); break
            if not csv:
                continue
            vec = _align_oof_by_id(csv, t)
            if sub.upper().startswith("CTB"): parts["CTB"] = vec
            if sub.upper().startswith("LGB"): parts["LGB"] = vec
            if sub.upper().startswith("XGB"): parts["XGB"] = vec
        if all(k in parts for k in ("CTB","LGB","XGB")):
            w = weights
            out[t] = w[0]*parts["CTB"] + w[1]*parts["LGB"] + w[2]*parts["XGB"]
    return out

OOF_CHEM = _maybe_load_oof_chemberta_auto()
OOF_FAST = _maybe_load_oof_fastsmiles_auto(weights=(0.4,0.3,0.3))

TEST_CHEM = pd.read_csv(CHEMBERTA_SUB) if CHEMBERTA_SUB and os.path.exists(CHEMBERTA_SUB) else None
TEST_FAST = pd.read_csv(FASTSMILES_SUB) if FASTSMILES_SUB and os.path.exists(FASTSMILES_SUB) else None

have_chem = (len(OOF_CHEM)==5) and (TEST_CHEM is not None)
have_fast = (len(OOF_FAST)==5) and (TEST_FAST is not None)
print(f"[Sources] Mordred ✅  ChemBERTa {'✅' if have_chem else '—'}  FastSMILES {'✅' if have_fast else '—'}")

MORDRED_OOF_FULL = {}
for t in TARGETS:
    full = np.full(len(train), np.nan, dtype=float)
    if t not in OOF_MORDRED:   # should not happen
        MORDRED_OOF_FULL[t] = full
        continue
    oof = OOF_MORDRED[t]
    smi = SMILES_NONNULL[t].astype(str).values
    # broadcast to all matching train rows (handles duplicates if any)
    for val, s in zip(oof, smi):
        mask = (train[SMI_COL].astype(str).values == s) & train[t].notna().values
        if mask.any():
            full[mask] = val
    MORDRED_OOF_FULL[t] = full

def best_weights_for_target_safe(t: str,
                                 sources: Dict[str, Tuple[np.ndarray, np.ndarray]],
                                 step: float = 0.02):
    names = list(sources.keys())
    y_true = train[t].values
    valid = ~np.isnan(y_true)

    keep = []
    for n in names:
        oof, _test = sources[n]
        if oof is not None and len(oof) == len(y_true):
            keep.append(n)
        else:
            print(f"[Skip {t}:{n}] OOF length {None if oof is None else len(oof)} != {len(y_true)}")
    if not keep:  # if all failed, fallback to equal average of test preds
        nn = list(sources.keys())
        w = {n: 1.0/len(nn) for n in nn}
        test_pred = np.mean([sources[n][1] for n in nn], axis=0)
        return w, test_pred

    names = keep
    oofs  = [sources[n][0] for n in names]
    tests = [sources[n][1] for n in names]

    mask = valid.copy()
    for o in oofs:
        mask &= ~np.isnan(o)

    yt = y_true[mask]
    oofs_m = [o[mask] for o in oofs]

    if len(names) == 1:
        return {names[0]: 1.0}, tests[0]

    ws = np.arange(0.0, 1.0 + 1e-9, step)
    best_mae, best_w = 1e9, None

    if len(names) == 2:
        for w0 in ws:
            w1 = 1.0 - w0
            pred = w0*oofs_m[0] + w1*oofs_m[1]
            mae = mean_absolute_error(yt, pred)
            if mae < best_mae:
                best_mae, best_w = mae, [w0, w1]
        test_pred = best_w[0]*tests[0] + best_w[1]*tests[1]
        return dict(zip(names, best_w)), test_pred

    if len(names) == 3:
        for w0 in ws:
            for w1 in ws:
                w2 = 1.0 - w0 - w1
                if w2 < 0:
                    continue
                pred = w0*oofs_m[0] + w1*oofs_m[1] + w2*oofs_m[2]
                mae = mean_absolute_error(yt, pred)
                if mae < best_mae:
                    best_mae, best_w = mae, [w0, w1, w2]
        test_pred = best_w[0]*tests[0] + best_w[1]*tests[1] + best_w[2]*tests[2]
        return dict(zip(names, best_w)), test_pred

    raise ValueError("Unexpected number of sources")

FINAL = {"id": test["id"].values}
WEIGHTS = {}

for t in TARGETS:
    # build test preds per source (aligned to test id)
    srcs: Dict[str, Tuple[np.ndarray, np.ndarray]] = {
        "mordred": (MORDRED_OOF_FULL[t], TEST_MORDRED[t]),
    }
    if have_chem:
        chem_oof = OOF_CHEM[t]
        chem_test = _align_submission_to_test(TEST_CHEM, t)
        srcs["chem"] = (chem_oof, chem_test)
    if have_fast:
        fast_oof = OOF_FAST[t]
        fast_test = _align_submission_to_test(TEST_FAST, t)
        srcs["fast"] = (fast_oof, fast_test)

    w, pred = best_weights_for_target_safe(t, srcs, step=0.02)
    WEIGHTS[t] = w

    if t == "FFV":
        pred = np.clip(pred, 0.0, 1.0)

    FINAL[t] = pred
    print(f"[Blend {t}] " + " + ".join(f"{w[k]:.2f}*{k}" for k in w))

sub = pd.DataFrame(FINAL, columns=["id","Tg","FFV","Tc","Density","Rg"])
sub.to_csv("/kaggle/working/submission.csv", index=False)
print("[Saved] /kaggle/working/submission.csv")


[Install] rdkit-2025.3.3-cp311-cp311-manylinux_2_28_x86_64.whl 
[Install] networkx-2.8.8-py3-none-any.whl 


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
scikit-image 0.25.2 requires networkx>=3.0, but you have networkx 2.8.8 which is incompatible.
nx-cugraph-cu12 25.2.0 requires networkx>=3.2, but you have networkx 2.8.8 which is incompatible.
torch 2.6.0+cu124 requires nvidia-cublas-cu12==12.4.5.8; platform_system == "Linux" and platform_machine == "x86_64", but you have nvidia-cublas-cu12 12.5.3.2 which is incompatible.
torch 2.6.0+cu124 requires nvidia-cuda-cupti-cu12==12.4.127; platform_system == "Linux" and platform_machine == "x86_64", but you have nvidia-cuda-cupti-cu12 12.5.82 which is incompatible.
torch 2.6.0+cu124 requires nvidia-cuda-nvrtc-cu12==12.4.127; platform_system == "Linux" and platform_machine == "x86_64", but you have nvidia-cuda-nvrtc-cu12 12.5.82 which is incompatible.
torch 2.6.0+cu124 requires nvidia-cuda-runtime-cu12==12.4.127; platform_

[Install] mordred-1.2.0-py3-none-any.whl --no-deps
[DATA_DIR] /kaggle/input/neurips-open-polymer-prediction-2025
tg shape: (9117, 594)
ffv shape: (7905, 597)
tc shape: (886, 566)
density shape: (1927, 473)
rg shape: (614, 545)
[Mordred] Computing test descriptors (2D; ignore_3D=True)...


100%|██████████| 3/3 [00:00<00:00,  3.57it/s]


desc_test shape: (3, 1613)
  [Tg] Mordred OOF MAE = 24.999101  (features: 593)
  [FFV] Mordred OOF MAE = 0.005934  (features: 596)
  [Tc] Mordred OOF MAE = 0.030800  (features: 565)
  [Density] Mordred OOF MAE = 0.071591  (features: 472)
  [Rg] Mordred OOF MAE = 1.570281  (features: 544)
[AUTO] CHEMBERTA_ROOT: /kaggle/input/neurips-opp2025-a-chemberta-t
[AUTO] FASTSMILES_ROOT: /kaggle/input/neurips-opp2025-a-fastsmiles-t
[AUTO] CHEMBERTA_SUB: /kaggle/input/neurips-opp2025-a-ensemble-0-068-lb/chemberta_submission.csv
[AUTO] FASTSMILES_SUB: /kaggle/input/neurips-opp2025-a-ensemble-0-068-lb/fastsmiles_submission.csv
[Sources] Mordred ✅  ChemBERTa ✅  FastSMILES ✅
[Skip Tg:chem] OOF length 9265 != 7973
[Skip Tg:fast] OOF length 9265 != 7973
[Blend Tg] 1.00*mordred
[Skip FFV:chem] OOF length 9265 != 7973
[Skip FFV:fast] OOF length 9265 != 7973
[Blend FFV] 1.00*mordred
[Skip Tc:chem] OOF length 9265 != 7973
[Skip Tc:fast] OOF length 9265 != 7973
[Blend Tc] 1.00*mordred
[Skip Density:chem] OOF